## RAG Ingestion 2 - Skip files that were already ingested
Every file gets a SHA-256 hash of its bytes, stored in the metadata of its chunks.
Before processing a file:

| Situation | Action |
|---|---|
| Same file name, same hash | **Skip** - nothing changed |
| New file name, but same hash as a stored file | **Skip** - duplicate content |
| New file name, new hash | **Process** - new file |
| Same file name, different hash | **Process** - re-ingest the whole file (notebook 3 does it chunk by chunk) |

### Step 1: Imports and configuration

In [23]:
import os
import re
import time
import hashlib
from dotenv import load_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone, ServerlessSpec

load_dotenv()

DATA_FOLDER = "Leave_Policy"
NAMESPACE = "rag-ingestion-dedup"   # own namespace so it doesn't mix with other notebooks' data
SUPPORTED = (".pdf", ".md", ".txt")

### Step 2: Connect to Pinecone (create the index if it doesn't exist)

In [24]:
pc = Pinecone(api_key=os.getenv("PineCone_API_Key"))
index_name = os.getenv("Pinecone_Index_Name")

if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
index = pc.Index(index_name)
print(index.describe_index_stats())

DescribeIndexStatsResponse(dimension=384, total_vector_count=0, metric='cosine', namespaces=0)


### Step 3: Read files
List the supported files in the folder and read their text (PDFs page by page).

In [ ]:
def list_files(folder):
    return sorted(
        os.path.join(folder, f) for f in os.listdir(folder) if f.lower().endswith(SUPPORTED)
    )


def file_id(path):
    """Pinecone-safe id prefix built from the file name."""
    return re.sub(r"[^A-Za-z0-9_.-]", "_", os.path.basename(path))


def load_pages(path):
    try:
        """Text of each PDF page, or the whole md/txt file as a single page."""
        if path.lower().endswith(".pdf"):
            return [p.page_content for p in PyPDFLoader(path).load()]
        with open(path, encoding="utf-8") as f:
            return [f.read()]
    except Exception as e:
        print(f"Error loading {path}: {e}")
        return []


files = list_files(DATA_FOLDER)
print("Files found:", [file_id(p) for p in files])

pages = load_pages(files[0])
print(f"\n{file_id(files[0])}: {len(pages)} pages")
print(pages[0][:300])

Files found: ['Leave_Policy_2015.pdf', 'Leave_Policy_2016.pdf', 'Leave_Policy_2017.pdf', 'Leave_Policy_2018.pdf', 'Leave_Policy_2019.pdf', 'Leave_Policy_2020.pdf', 'Leave_Policy_2021.pdf', 'Leave_Policy_2022.pdf', 'Leave_Policy_2023.pdf', 'Leave_Policy_2024.pdf', 'Leave_Policy_2025.pdf', 'Leave_Policy_2026.pdf']

Leave_Policy_2015.pdf: 3 pages
Acme Technologies India Pvt. Ltd. | Leave Policy 2015 | HR-LV-2015
Page 1
 Leave Policy 2015
 Acme Technologies India Pvt. Ltd.
Field
Details
Document ID
HR-LV-2015
Policy year
2015
Effective period
1 January 2015 to 31 December 2015
Supersedes
None (first consolidated policy)
Owner
Human Resources 


### Step 4: Chunking
Split each page into chunks, remembering the page number.

In [26]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", " ", ""],
)


def chunk_file(path):
    """List of (chunk_text, page_no) for one file."""
    chunks = []
    for page_no, page in enumerate(load_pages(path), start=1):
        chunks += [(text, page_no) for text in splitter.split_text(page)]
    return chunks


sample_chunks = chunk_file(files[0])
print(f"{file_id(files[0])}: {len(sample_chunks)} chunks")
for text, page_no in sample_chunks[:3]:
    print(f"\n--- page {page_no}, length {len(text)} ---")
    print(text)

Leave_Policy_2015.pdf: 15 chunks

--- page 1, length 410 ---
Acme Technologies India Pvt. Ltd. | Leave Policy 2015 | HR-LV-2015
Page 1
 Leave Policy 2015
 Acme Technologies India Pvt. Ltd.
Field
Details
Document ID
HR-LV-2015
Policy year
2015
Effective period
1 January 2015 to 31 December 2015
Supersedes
None (first consolidated policy)
Owner
Human Resources Department
Applies to
All permanent employees of Acme Technologies India Pvt. Ltd.
1. Purpose and Scope (2015)

--- page 1, length 476 ---
1. Purpose and Scope (2015)
This document sets out the Leave Policy of Acme Technologies India Pvt. Ltd. for the calendar year 2015.
It applies to all permanent employees in all offices and covers casual, sick, earned, maternity, paternity
and other leave. The leave year runs from 1 January 2015 to 31 December 2015. This policy is valid only
for leave taken in 2015; leave taken in other years is governed by the policy of that year.
2. Summary of Changes in the 2015 Policy

--- page 1, length 469

### Step 5: Embedding
Convert chunk text into 384-dimension vectors.

In [27]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


def embed(texts):
    return model.encode(texts)


sample_vectors = embed([text for text, _ in sample_chunks[:3]])
print("Vector shape:", sample_vectors.shape)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1145.90it/s]


Vector shape: (3, 384)


### Step 6: Insert into Pinecone
Upsert records in batches of 100.

In [28]:
def upsert_batches(records, batch_size=100):
    for start in range(0, len(records), batch_size):
        index.upsert(vectors=records[start:start + batch_size], namespace=NAMESPACE)

### Step 7: File hashing
Same bytes -> same hash, any change -> different hash.

In [29]:
def file_hash(path):
    with open(path, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()


# file1 : get haskey  hash_1234


#r,rb,w,wb
#r: read text
#rb: read binary
#w: write text
#wb: write binary

print(file_id(files[0]), "->", file_hash(files[0]))

Leave_Policy_2015.pdf -> 1dad8c5c2b367a2f3908aed8c5fc434a59a056f5fba0f8cf3a5daf095d4907b5


### Step 8: Check what Pinecone already has for a file

In [ ]:
def stored_hash(fid):
    """file_hash saved on chunk #0 of this file, or None if the file was never ingested."""
    res = index.fetch(ids=[f"{fid}#0"], namespace=NAMESPACE)
    v = res.vectors.get(f"{fid}#0")
    return v.metadata.get("file_hash") if v else None


PROBE = embed("probe").tolist()  # any vector works, the filter does the matching


def stored_under_other_name(h, fid):
    """Name of another file already stored with exactly the same content."""
    res = index.query(
        vector=PROBE,
        top_k=1,
        include_metadata=True,
        namespace=NAMESPACE,
        filter={"file_hash": {"$eq": h}, "source": {"$ne": fid}},
    )
    return res.matches[0].metadata["source"] if res.matches else None


def existing_ids(fid):
    # index.list() yields one page at a time; the ids are in page.vectors
    return [item.id for page in index.list(prefix=f"{fid}#", namespace=NAMESPACE) for item in page.vectors]

### Step 9: Ingest one file (chunk + embed + insert, hash stored in metadata)

In [31]:
def ingest_file(path, fid, h):
    chunks = chunk_file(path)
    vectors = embed([text for text, _ in chunks])

    records = [
        {
            "id": f"{fid}#{i}",
            "values": vectors[i].tolist(),
            "metadata": {
                "text": text, "source": fid, "page": page_no,
                "chunk_index": i, "file_hash": h,
            },
        }
        for i, (text, page_no) in enumerate(chunks)
    ]
    # Chunk #0 carries the hash we check next time, so write it last:
    # if a run is interrupted, the file is picked up again on the next run
    upsert_batches(records[1:])
    upsert_batches(records[:1])
    return len(records)

### Step 10: Decide - skip, process new, or re-process changed file

In [32]:
def ingest_if_new(path):
    fid, h = file_id(path), file_hash(path)
    old = stored_hash(fid)

    if old == h:
        return f"SKIPPED  {fid}: same file already ingested"

    if old is None:
        dup = stored_under_other_name(h, fid)
        if dup:
            return f"SKIPPED  {fid}: same content as {dup}"

    old_ids = existing_ids(fid) if old else []
    n = ingest_file(path, fid, h)

    # New version may have fewer chunks: drop the leftover ids of the old version
    stale = [i for i in old_ids if int(i.rsplit("#", 1)[1]) >= n]
    if stale:
        index.delete(ids=stale, namespace=NAMESPACE)

    status = "NEW     " if old is None else "CHANGED "
    return f"{status} {fid}: inserted {n} chunks, removed {len(stale)} old chunks"


def ingest_folder(folder):
    for path in list_files(folder):
        print(ingest_if_new(path))

### Step 11: First run - every file is new

In [33]:
ingest_folder(DATA_FOLDER)

NEW      Leave_Policy_2015.pdf: inserted 15 chunks, removed 0 old chunks
NEW      Leave_Policy_2016.pdf: inserted 17 chunks, removed 0 old chunks
NEW      Leave_Policy_2017.pdf: inserted 17 chunks, removed 0 old chunks
NEW      Leave_Policy_2018.pdf: inserted 18 chunks, removed 0 old chunks
NEW      Leave_Policy_2019.pdf: inserted 19 chunks, removed 0 old chunks
NEW      Leave_Policy_2020.pdf: inserted 19 chunks, removed 0 old chunks
NEW      Leave_Policy_2021.pdf: inserted 19 chunks, removed 0 old chunks
NEW      Leave_Policy_2022.pdf: inserted 21 chunks, removed 0 old chunks
NEW      Leave_Policy_2023.pdf: inserted 20 chunks, removed 0 old chunks
NEW      Leave_Policy_2024.pdf: inserted 20 chunks, removed 0 old chunks
NEW      Leave_Policy_2025.pdf: inserted 21 chunks, removed 0 old chunks
NEW      Leave_Policy_2026.pdf: inserted 20 chunks, removed 0 old chunks


### Step 12: Second run - same files, so everything is skipped

In [20]:
time.sleep(5)  # freshly written records take a few seconds to become visible
ingest_folder(DATA_FOLDER)

SKIPPED  Leave_Policy_2015.pdf: same file already ingested
SKIPPED  Leave_Policy_2016.pdf: same file already ingested
SKIPPED  Leave_Policy_2017.pdf: same file already ingested
SKIPPED  Leave_Policy_2018.pdf: same file already ingested
SKIPPED  Leave_Policy_2019.pdf: same file already ingested
SKIPPED  Leave_Policy_2020.pdf: same file already ingested
SKIPPED  Leave_Policy_2021.pdf: same file already ingested
SKIPPED  Leave_Policy_2022.pdf: same file already ingested
SKIPPED  Leave_Policy_2023.pdf: same file already ingested
SKIPPED  Leave_Policy_2024.pdf: same file already ingested
SKIPPED  Leave_Policy_2025.pdf: same file already ingested
SKIPPED  Leave_Policy_2026.pdf: same file already ingested


### Step 13: Same content under a different file name is skipped too

In [21]:
import shutil

DEMO_FOLDER = "RAG_Dedup_Demo"
# os.makedirs(DEMO_FOLDER, exist_ok=True)
# shutil.copy(os.path.join(DATA_FOLDER, "Leave_Policy_2026.pdf"),
#             os.path.join(DEMO_FOLDER, "Leave_Policy_2026_copy.pdf"))

ingest_folder(DEMO_FOLDER)

SKIPPED  Leave_Policy_2015.pdf: same file already ingested
SKIPPED  Leave_Policy_2016.pdf: same file already ingested
SKIPPED  Leave_Policy_2017.pdf: same file already ingested
SKIPPED  Leave_Policy_2018.pdf: same file already ingested
SKIPPED  Leave_Policy_2019.pdf: same file already ingested
SKIPPED  Leave_Policy_2020.pdf: same file already ingested
SKIPPED  Leave_Policy_2021.pdf: same file already ingested
SKIPPED  Leave_Policy_2022.pdf: same file already ingested
SKIPPED  Leave_Policy_2023.pdf: same file already ingested
SKIPPED  Leave_Policy_2024.pdf: same file already ingested
SKIPPED  Leave_Policy_2025.pdf: same file already ingested
SKIPPED  Leave_Policy_2026.pdf: same file already ingested


### Step 14: Retrieval check

In [22]:
def search(question, top_k=3):
    res = index.query(
        vector=embed(question).tolist(),
        top_k=top_k,
        include_metadata=True,
        namespace=NAMESPACE,
    )
    for m in res.matches:
        print(f"\nScore: {m.score:.3f} | {m.metadata['source']} (page {m.metadata['page']})")
        print(m.metadata["text"])


search("How many sick leaves are available in 2026?", top_k=3)


Score: 0.467 | Leave_Policy_2026.pdf (page 4)
- Negative leave balances are not permitted.
- Unauthorised absence of more than 3 consecutive days may lead to disciplinary action.
- HR may amend this policy with notice. The management decision is final in case of dispute.
13. Frequently Asked Questions (2026 Policy)
Q: How many casual leaves does an employee get in 2026?
A: 12 days of Casual Leave per year.
Q: How many sick leaves are there in 2026?
A: 12 days per year; medical certificate needed when it exceeds 2 days.

Score: 0.451 | Leave_Policy_2024.pdf (page 4)
13. Frequently Asked Questions (2024 Policy)
Q: How many casual leaves does an employee get in 2024?
A: 12 days of Casual Leave per year.
Q: How many sick leaves are there in 2024?
A: 10 days per year; medical certificate needed when it exceeds 2 days.
Q: How many earned leaves can be carried forward from 2024?
A: Up to 45 days.
Q: What is the maternity leave duration in 2024?
A: 26 weeks for the first two children and 12 w